In [1]:
#Inicialización 
import xrfclk
import xrfdc
import pynq
from pynq import Overlay, MMIO
from pynq import lib
import numpy as np
import time
import os
import subprocess
import time 

In [2]:
CLOCKWIZARD_LOCK_ADDRESS = 0x0004
CLOCKWIZARD_RESET_ADDRESS = 0x0000
CLOCKWIZARD_RESET_TOKEN = 0x000A
MTS_START_TILE = 0x01
MAX_DAC_TILES = 4
MAX_ADC_TILES = 4
DAC_REF_TILE = 2
ADC_REF_TILE = 2

RFSOC4X2_LMK_FREQ = 500.0
RFSOC4X2_LMX_FREQ = 500.0
RFSOC4X2_DAC_TILES = 0b0101
RFSOC4X2_ADC_TILES = 0b0101

In [3]:
xrfclk.set_ref_clks(lmk_freq = RFSOC4X2_LMK_FREQ, lmx_freq = RFSOC4X2_LMX_FREQ) # Cargar la configuración deseada
#xrfclk.set_ref_clks(lmk_freq = 245.76, lmx_freq = 491.52) # Cargar la configuración base 
#xrfclk.set_ref_clks(lmk_freq = 245.76, lmx_freq = 409.6) # Cargar otra configuración 

In [4]:
# Comprobación de que no haya un Overlay ya cargado
board = os.getenv('BOARD') 
# Run lsmod command to get the loaded modules list
output = subprocess.check_output(['lsmod'])
# Check if "zocl" is present in the output
if b'zocl' in output:
    # If present, remove the module using rmmod command
    rmmod_output = subprocess.run(['rmmod', 'zocl'])
    # Check return code
    assert rmmod_output.returncode == 0, "Could not restart zocl. Please Shutdown All Kernels and then restart"
    # If successful, load the module using modprobe command
    modprobe_output = subprocess.run(['modprobe', 'zocl'])
    assert modprobe_output.returncode == 0, "Could not restart zocl. It did not restart as expected"
else:
    modprobe_output = subprocess.run(['modprobe', 'zocl'])
    # Check return code
    assert modprobe_output.returncode == 0, "Could not restart ZOCL!"

In [5]:
# Cargamos el Overlay
ol = Overlay('/usr/local/share/pynq-venv/lib/python3.10/site-packages/pynq/overlays/LB_Ext/DIS.bit',ignore_version=True)

In [6]:
#ol? # Verifica el estado del Overlay

In [7]:
ol.ACTIVE_DAC_TILES = RFSOC4X2_DAC_TILES
ol.ACTIVE_ADC_TILES = RFSOC4X2_ADC_TILES

In [8]:
ol.xrfdc = ol.usp_rf_data_converter_0

In [9]:
ol.xrfdc.mts_dac_config.RefTile = DAC_REF_TILE  # DAC tile distributing reference clock
ol.xrfdc.mts_adc_config.RefTile = ADC_REF_TILE  # ADC 

In [10]:
#INIT SYNC TILES

In [11]:
ol.xrfdc.mts_dac_config.Tiles = 0b0001 # turn only one tile on first
ol.xrfdc.mts_adc_config.Tiles = 0b0001
ol.xrfdc.mts_dac_config.SysRef_Enable = 1
ol.xrfdc.mts_adc_config.SysRef_Enable = 1
ol.xrfdc.mts_dac_config.Target_Latency = -1
ol.xrfdc.mts_adc_config.Target_Latency = -1

In [12]:
ol.xrfdc.mts_dac()
ol.xrfdc.mts_adc()

In [13]:
ol.MTS_clkwiz.mmio.write_reg(CLOCKWIZARD_RESET_ADDRESS, CLOCKWIZARD_RESET_TOKEN)

In [14]:
time.sleep(0.1)
# Reset only user selected DAC tiles
bitvector = ol.ACTIVE_DAC_TILES
for n in range(MAX_DAC_TILES):
    if (bitvector & 0x1):
        ol.xrfdc.dac_tiles[n].Reset()
    bitvector = bitvector >> 1
# Reset ADC FIFO of only user selected tiles - restarts MTS engine
for toggleValue in range(0,1):
    bitvector = ol.ACTIVE_ADC_TILES
    for n in range(MAX_ADC_TILES):
        if (bitvector & 0x1):
            ol.xrfdc.adc_tiles[n].SetupFIFOBoth(toggleValue)
        bitvector = bitvector >> 1

In [15]:
#SYNC TILES
dacTarget=-1
adcTarget=-1

In [16]:
if ol.ACTIVE_DAC_TILES > 0:
    ol.xrfdc.mts_dac_config.Tiles = ol.ACTIVE_DAC_TILES # group defined in binary 0b1111
    ol.xrfdc.mts_dac_config.SysRef_Enable = 1
    ol.xrfdc.mts_dac_config.Target_Latency = dacTarget 
    ol.xrfdc.mts_dac()
else:
    ol.xrfdc.mts_dac_config.Tiles = 0x0
    ol.xrfdc.mts_dac_config.SysRef_Enable = 0

In [17]:
if ol.ACTIVE_ADC_TILES > 0:
    ol.xrfdc.mts_adc_config.Tiles = ol.ACTIVE_ADC_TILES
    ol.xrfdc.mts_adc_config.SysRef_Enable = 1
    ol.xrfdc.mts_adc_config.Target_Latency = adcTarget
    ol.xrfdc.mts_adc()
else:
    ol.xrfdc.mts_adc_config.Tiles = 0x0
    ol.xrfdc.mts_adc_config.SysRef_Enable = 0

In [18]:
Pulso=ol.Tx_Concats.Muestras_DDFS_Frec_c_0
Pulso.write(0x0,200) #Indicamos que se genere un pulso de 100 MHz
Mux_cal=ol.Tx_Concats.Mux_Cal_0
Mux_cal.write(0x0,1) # Inyectamos el pulso 

In [19]:
coeff_struct = xrfdc._ffi.new("XRFdc_Calibration_Coefficients*")
#`0` (OCB1), `1` (OCB2), `2` (GCB), o `3` (TSCB)
xrfdc._lib.XRFdc_GetCalCoefficients(ol.xrfdc._instance, 2, 0, 3, coeff_struct)
print('Coeff7:', coeff_struct.Coeff7)
#xrfdc._lib.XRFdc_GetCalCoefficients(ol.xrfdc._instance, 2, 1, 0, coeff_struct)

Coeff7: 33488897


In [20]:


#xrfdc._lib.XRFdc_SetCalCoefficients(ol.xrfdc._instance, Tile226, ADC_A,cal_type, cal_coeffs)
#xrfdc._lib.XRFdc_SetCalCoefficients(ol.xrfdc._instance, Tile226, ADC_B,cal_type, cal_coeffs)

In [21]:
Mux_cal.write(0x0,0)

In [22]:
# Calibrado
# Puesta en marcha 
Tile226=2
ADC_A=0
ADC_B=1

Pulso=ol.Tx_Concats.Muestras_DDFS_Frec_c_0
Pulso.write(0x0,100) #Indicamos que se genere un pulso de 100 MHz
Mux_cal=ol.Tx_Concats.Mux_Cal_0
Mux_cal.write(0x0,1) # Inyectamos el pulso 


modo2 = 2
xrfdc._lib.XRFdc_SetCalibrationMode(ol.xrfdc._instance, Tile226, ADC_A, modo2)
xrfdc._lib.XRFdc_SetCalibrationMode(ol.xrfdc._instance, Tile226, ADC_B, modo2)



freeze_settings = xrfdc._ffi.new("XRFdc_Cal_Freeze_Settings *")
freeze_settings.CalFrozen = 0      # 0 = Activar calibración
freeze_settings.DisableFreezePin = 1
freeze_settings.FreezeCalibration = 0 #3

xrfdc._lib.XRFdc_SetCalFreeze(ol.xrfdc._instance, Tile226, ADC_A, freeze_settings)
xrfdc._lib.XRFdc_SetCalFreeze(ol.xrfdc._instance, Tile226, ADC_B, freeze_settings)

ol.xrfdc.adc_tiles[Tile226].blocks[ADC_A].UpdateEvent(2) # Evento de actualización
ol.xrfdc.adc_tiles[Tile226].blocks[ADC_B].UpdateEvent(2)

print(ol.xrfdc.adc_tiles[Tile226].blocks[ADC_A].BlockStatus)
print(ol.xrfdc.adc_tiles[Tile226].blocks[ADC_B].BlockStatus)

{'SamplingFreq': 4.0, 'AnalogDataPathStatus': 1, 'DigitalDataPathStatus': 32, 'DataPathClocksStatus': 1, 'IsFIFOFlagsEnabled': 3, 'IsFIFOFlagsAsserted': 0}
{'SamplingFreq': 4.0, 'AnalogDataPathStatus': 1, 'DigitalDataPathStatus': 32, 'DataPathClocksStatus': 1, 'IsFIFOFlagsEnabled': 3, 'IsFIFOFlagsAsserted': 0}


In [23]:
# Verificación coeficientes
cal_blocks = ['OCB1','OCB2','GCB','TSCB']
for i, name in enumerate(cal_blocks):
    coeff = xrfdc._ffi.new("XRFdc_Calibration_Coefficients*")
    xrfdc._lib.XRFdc_GetCalCoefficients(ol.xrfdc._instance, Tile226, ADC_A, i, coeff)
    print(f"{name}: {xrfdc._unpack_value(coeff)}")

OCB1: {'Coeff0': 4291231706, 'Coeff1': 4292542433, 'Coeff2': 589829, 'Coeff3': 1310747, 'Coeff4': 0, 'Coeff5': 0, 'Coeff6': 0, 'Coeff7': 0}
OCB2: {'Coeff0': 5046193, 'Coeff1': 57410235, 'Coeff2': 4290576466, 'Coeff3': 4243324256, 'Coeff4': 0, 'Coeff5': 0, 'Coeff6': 0, 'Coeff7': 0}
GCB: {'Coeff0': 15466496, 'Coeff1': 14745635, 'Coeff2': 14024724, 'Coeff3': 15667181, 'Coeff4': 0, 'Coeff5': 0, 'Coeff6': 0, 'Coeff7': 0}
TSCB: {'Coeff0': 33488897, 'Coeff1': 33488897, 'Coeff2': 33488897, 'Coeff3': 33488897, 'Coeff4': 33488897, 'Coeff5': 33488897, 'Coeff6': 33488897, 'Coeff7': 33488897}


In [24]:
# Parada
time.sleep(30)

freeze_settings.CalFrozen = 1     # 1 = Desactivar calibración
freeze_settings.FreezeCalibration = 7
xrfdc._lib.XRFdc_SetCalFreeze(ol.xrfdc._instance, Tile226, ADC_A, freeze_settings)
xrfdc._lib.XRFdc_SetCalFreeze(ol.xrfdc._instance, Tile226, ADC_B, freeze_settings)

ol.xrfdc.adc_tiles[Tile226].blocks[ADC_A].UpdateEvent(2) # Evento de actualización
ol.xrfdc.adc_tiles[Tile226].blocks[ADC_B].UpdateEvent(2)
Mux_cal.write(0x0,0) # Inyectamos nuestros datos

metal: error:     
 Invalid FreezeCalibration option (7) for ADC 2 block 0 in XRFdc_SetCalFreeze
metal: error:     
 Invalid FreezeCalibration option (7) for ADC 2 block 1 in XRFdc_SetCalFreeze


In [6]:
Mux=ol.Mux_and_Data.Multiplexor_0 

In [13]:
Mux.write(0x0,0) #Datos de entrada al modelo Tx (0: patrón 11001100; 2: contador de 8 bits)

In [8]:
Rx=ol.Rx_and_ILAs.Rx_monitorizado_0

In [9]:
Rx.write(0x8,5) #5 Umbral

In [10]:
# Desfase manual 
Rx.write(0xc,4) #4

In [11]:
#Delays variables
#ambos de base tienen que ser 4
Rx.write(0x10,3) #Delay de sincronización del valid con la señal antes de la demodulación.
Rx.write(0x0,3) #Delay que bypasea la detección.

In [12]:
Rx.write(0x4,0) #Multiplicamos la señal por -1 (1) o no (0)